In [ ]:
pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)
pl.Config.set_fmt_str_lengths(200)

# Cadrage et segmentation

Ce premier notebook pose le cadre de l'étude et isole notre population cible avant de la segmenter pour modéliser la PD.

**Étapes suivies :**
- **1. Cadrage :** exploration de la base, représentativité, temporalité de la cible, critères d'exclusion et traitement des dossiers joints.
- **2. Segmentation :** sélection univariée des variables discriminantes, segmentation par arbre CART, validation de l'hétérogénéité des risques (analyse d'IV) et arbitrage.
- **3. Analyse descriptive :** comparaison des segments retenus.


In [ ]:
from pathlib import Path

import pandas as pd
import polars as pl

ID, TARGET, DATE = "gdt", "ddefaut_ndb", "datdelhis"
SOURCE = Path("data/_01_99__ensai_par_final.sas7bdat.gz")
CACHE = Path("data/processed/base.parquet")

Chargement : le fichier SAS est converti une seule fois en parquet (lecture plus rapide ensuite). Les `NaN` issus de SAS sont convertis en `null` pour être comptés comme manquants, et `datdelhis` (YYYYMM) est converti en date.

In [ ]:
from utils.exploration import nan_to_null

if not CACHE.exists():
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    pl.from_pandas(pd.read_sas(SOURCE, encoding="latin1")).write_parquet(CACHE)

# gestion des valeurs manquantes
base = nan_to_null(pl.read_parquet(CACHE)).with_columns(
    pl.date(pl.col(DATE).cast(pl.Int32) // 100, pl.col(DATE).cast(pl.Int32) % 100, 1).alias(DATE)
)
base.shape

# Partie 1 — Cadrage de la population
## 1.1 Structure de la base

In [ ]:
print("lignes :", base.height)
print(f"{ID} distincts :", base[ID].n_unique())
print(f"{ID} x {DATE} distincts :", base.select(ID, DATE).n_unique())
print("dates d'observation :", base[DATE].unique().sort().to_list())
print(base.group_by(ID).agg(nb_mois=pl.len()).group_by("nb_mois").agg(nb_clients=pl.len()).sort("nb_mois"))

- `gdt × datdelhis` est unique : une ligne = un dossier observé à une date.
- Seulement **4 mois d'observation** (janvier à avril 2021) :
  - la stabilité temporelle se mesure sur ces 4 mois ;
  - l'échantillon Out-of-Time sera **202104** (avril 2021) ;
  - les 4 mois, avec leur fenêtre de défaut à 12 mois, couvrent une **seule année de défauts** (2021, période Covid atypique). La LRA, moyenne sur un cycle complet (≥ 5 ans), n'est donc pas estimable sur la base : au calibrage, on retiendra une approximation avec une MoC de catégorie A.
- La grande majorité des dossiers n'apparaît qu'**un seul mois** : c'est un échantillon mensuel, pas un panel. Le découpage apprentissage / test se fera néanmoins **par `gdt`**, pour qu'aucun dossier ne soit des deux côtés.


## 1.2 Plan de sondage

Le défaut étant un événement rare, la base est issue d'un **sous-échantillonnage stratifié** (conservation de tous les défauts et d'une fraction des clients sains) :

* **`SelectionProb` ($\pi_i$)** : Probabilité qu'avait l'observation d'être retenue dans la base ($\approx 1$ pour les défauts, $< 1$ pour les sains).
* **`SamplingWeight` ($w_i$)** : Poids de sondage ($w_i = 1 / \pi_i$). Indique le nombre de clients réels du portefeuille représentés par cette ligne.

**Impacts clés sur le projet :**
1. **Taux de défaut brut** : Artificiellement gonflé dans la base ; calculer la moyenne pondérée par `SamplingWeight` pour retrouver le taux réel.
2. **Modélisation (Gini / Scorecard)** : L'ordonnancement et les coefficients $\beta$ ne sont pas biaisés par ce plan de sondage.
3. **Calibrage (PD réelle)** : Nécessite de redresser les probabilités ou de corriger l'intercept ($\beta_0$).

 On va donc vérifier si ce tirage a déformé la proportion de défauts.


In [ ]:
print(base.group_by(TARGET).agg(
    n=pl.len(),
    n_poids_distincts=pl.col("SamplingWeight").n_unique(),
    poids_min=pl.col("SamplingWeight").min(),
    poids_max=pl.col("SamplingWeight").max(),
))
print("taux brut    :", base[TARGET].mean())
print("taux pondéré :", base.select((pl.col("SamplingWeight") * pl.col(TARGET)).sum() / pl.col("SamplingWeight").sum()).item())

Les poids de sondage valent tous environ 66,67, que ce soit pour les sains ou pour les défauts. Ça correspond à un tirage d'environ 1,5 % de la population (1 / 66,67). On trouve 4 valeurs de poids différentes par classe, ce qui fait 8 strates : le tirage a vraisemblablement été stratifié par mois et par statut (sain / défaut), avec le même taux de tirage dans chaque strate (allocation proportionnelle). Les défauts n'ont donc pas été sur-représentés, contrairement à ce qu'on fait souvent quand la cible est rare.

**Conséquence pour le calibrage**

Le taux de défaut brut (0,781 %) et le taux pondéré (0,781 %) sont quasiment identiques : l'échantillon est représentatif du portefeuille sur la période observée. On n'aura donc pas besoin de corriger l'intercept au moment du calibrage (ch. 1.3).

**Exclusion de `SelectionProb` et `SamplingWeight`**

Les poids des défauts (entre 66,564 et 66,657) et ceux des sains (entre 66,666 et 66,667) ne se chevauchent pas : connaître le poids d'une ligne suffit à savoir si c'est un défaut. Ces deux variables contiennent donc directement l'information de la cible (fuite). On les exclut de toute la suite : segmentation, sélection de variables et modèles.


## 1.3 La cible dans le temps
`ddefaut_ndb` = passage en défaut dans les 12 mois suivant la date d'observation.

In [ ]:
from utils.exploration import risk_rate_by_modality, chi2_association

print(risk_rate_by_modality(base, DATE, TARGET).sort(DATE))

avril = pl.col(DATE) == pl.date(2021, 4, 1)
print(chi2_association(base.with_columns(avril=avril), "avril", TARGET))


Les volumes sont stables d'un mois à l'autre (environ 78 000 lignes, soit un quart de la base chaque mois). Le taux de défaut est stable de janvier à mars (autour de 0,75 %), puis monte à 0,87 % en avril. Avec le taux de janvier-mars, on s'attendait à environ 588 défauts en avril, on en observe 679 : l'écart est significatif (test du Chi², p == 0.0015074549 < 0,05).

- Si la fenêtre de 12 mois était incomplète pour les derniers mois, le taux baisserait en fin de période. Ici il augmente, donc la cible semble bien complète.
- La hausse d'avril vient surtout des dossiers sans engagement.
- À garder en tête : l'échantillon OOT (avril) aura un taux de défaut plus élevé que l'échantillon d'apprentissage. Ça ne devrait pas beaucoup jouer sur le Gini, qui mesure l'ordonnancement et pas le niveau, mais il faudra en tenir compte au calibrage.


## 1.4 Définition du périmètre
### Variables de périmètre constantes

In [ ]:
for v in ["type_PM_PP", "code_marche_v2"]:
    print(risk_rate_by_modality(base, v, TARGET))

Les deux variables n'ont qu'une seule modalité, sans valeur manquante : tous les dossiers sont des personnes physiques (PP) du marché des particuliers. Il n'y a aucun filtre à ajouter. Comme ces deux variables sont constantes, elles n'apportent aucune information : on les retire de la modélisation.

### Dossiers déjà en défaut à la date d'observation
Un dossier déjà en défaut au moment de l'observation n'a pas de probabilité de défaut à estimer, puisque le défaut a déjà eu lieu. Si on le garde, le modèle apprend à reconnaître des défauts existants au lieu de les prédire, ce qui gonfle artificiellement le Gini. On regarde donc quelles variables permettent de repérer ces dossiers.


In [ ]:
from utils.exploration import controle_groupes

print(base.select(
    pl.col("DATE_DERNIER_DFO").min().alias("dernier_dfo_min"),
    pl.col("DATE_DERNIER_DFO").max().alias("dernier_dfo_max"),
    pl.col(DATE).min().alias("observation_min"),
))

volumes, recouvrement = controle_groupes(base, {
    "arr_90j": pl.col("NBJ_ARR_SS") >= 90,
    "dfo_non_resolu": pl.col("DATE_DERNIER_DFO") >= pl.col("DATE_SAIN"),
    "date_sain_null": pl.col("DATE_SAIN").is_null(),
    "dfo_M1_M3": pl.col("Dfo_max_Mm1_Mm3") == 1,
}, TARGET)
print(volumes)
print(recouvrement)

**Constat**
- `NBJ_ARR_SS ≥ 90` (au moins 90 jours d'arriérés significatifs) : c'est la définition réglementaire du défaut. Sur ces 170 dossiers, 72 % sont en défaut dans l'année. Ce n'est pas 100 %, probablement parce que certains régularisent leur situation ou parce que la cible ne compte que les nouvelles entrées en défaut. Ils représentent 5 % des défauts pour seulement 0,05 % de la population : on les **exclut**.
- `Dfo_max_Mm1_Mm3 = 1` (défaut au cours des 3 derniers mois) : 25 % de ces 452 dossiers font défaut dans l'année. C'est beaucoup plus que la moyenne (0,78 %), mais loin de 100 % : ce ne sont pas des défauts en cours, plutôt des dossiers récemment sortis du défaut. Cette information est connue à la date d'observation, donc on les **garde** ; la variable servira de risk driver.
- `DATE_SAIN` manquante (14 lignes) et défaut non résolu (2 lignes) : très peu de lignes et aucun défaut, on les garde.
- `DATE_DERNIER_DFO` ne peut pas servir à repérer les défauts à date : elle s'arrête au 31/12/2020, avant la première observation. D'ailleurs, aucun des 170 dossiers à 90 jours d'arriérés ou plus n'y apparaît comme en défaut. Le seul indicateur fiable est donc `NBJ_ARR_SS`.



### Banque Privée / Gestion de Fortune

In [ ]:
print(risk_rate_by_modality(base, "cod_axe_unite_max", TARGET).sort("cod_axe_unite_max"))

**Constat** : Banque Privée (2) et Gestion de Fortune (3) représentent ≈ 3,7 % des dossiers et ≈ 35 défauts. Ce sont bien des particuliers, et un modèle PD doit couvrir tout le portefeuille → **gardés**. Trop peu de défauts pour un segment propre : ce sera un risk driver (modalités 2 et 3 regroupées).

### Mineurs

In [ ]:
from utils.exploration import controle_groupes

AGE_ANS = pl.coalesce("AGE_PP_EN_MOIS", "AGE_PP_EN_MOIS_med") / 12  # âge des dossiers joints dans _med 

volumes, _ = controle_groupes(base, {"mineurs": AGE_ANS < 18}, TARGET)
print(volumes)

**Constat** : 48 407 dossiers sont des mineurs, soit 15,5 % de la base, pour seulement 24 défauts (0,05 %, seize fois moins que la moyenne). Un mineur n'a pas la capacité juridique de contracter un crédit, donc il n'y a pas vraiment de risque de crédit à modéliser pour lui. Les garder ajouterait beaucoup de dossiers sains très faciles à classer, ce qui gonflerait le Gini sans rien apporter : on les **exclut** du périmètre (ils seraient notés par une règle simple).

L'âge reste par contre une variable candidate pour le modèle. On décidera de l'utiliser ou non lors de la sélection de variables, en comparant le Gini avec et sans l'âge, car c'est un critère sensible du point de vue de la non-discrimination.

### Application des filtres


In [ ]:
from utils.exploration import appliquer_filtres

base_perim, suivi_perimetre = appliquer_filtres(base, {
    "défaut à date (NBJ_ARR_SS >= 90)": pl.col("NBJ_ARR_SS") >= 90,
    "mineurs ": AGE_ANS < 18,
}, TARGET)
print(suivi_perimetre)

## 1.5 Dossiers joints (`nb_tiers > 1`)
Pour les dossiers à plusieurs clients, les variables individuelles (âge, ancienneté, comportement de compte…) ne sont disponibles que sous forme agrégée : `_min`, `_max`, `_med`. On vérifie le remplissage.

In [ ]:
VARS_TEST = ["AGE_PP_EN_MOIS", "ANCIENNETE_modif", "CRTAD_IND_0042", "ENCOURS_PAR"]
print(base_perim.group_by(joint=pl.col("nb_tiers") > 1).agg(
    pl.len().alias("n"),
    *[pl.col(v).is_not_null().mean().alias(f"{v}_rempli") for v in VARS_TEST],
    *[pl.col(f"{v}_med").is_not_null().mean().alias(f"{v}_med_rempli") for v in VARS_TEST],
).sort("joint"))

Pour un dossier qui regroupe plusieurs clients (par exemple un couple), les variables propres à chaque personne (âge, ancienneté, comportement de compte…) ne sont pas renseignées directement. La base donne à la place le minimum, le maximum et la médiane des membres du dossier (`_min`, `_max`, `_med`). Le tableau le confirme : les dossiers individuels ont toujours la variable de base et jamais la version `_med`, et c'est l'inverse pour les dossiers joints. Si on ne fait rien, tous les dossiers joints apparaîtront comme « manquants » sur ces variables, et la classe « manquant » voudra en fait dire « dossier joint ».

**Ce qu'on fait** : on regroupe les deux en une seule colonne, en prenant la valeur individuelle quand elle existe et la médiane du dossier sinon (fonction `coalesce`). Pour un couple, la médiane correspond à la moyenne des deux membres.
- Trois codes n'ont pas de médiane, seulement un min et un max : les deux indicateurs d'interdit bancaire et l'axe de clientèle. Une médiane n'aurait pas de sens pour un code : on prend le max, par prudence. Si un membre du couple est interdit bancaire, on considère que le dossier l'est.
- SAS limite les noms de variables à 32 caractères, donc certaines versions `_med` ont un nom raccourci (ex. `RATIO_SOLDE_MOYEN_SUR_EPAR_med` pour `RATIO_SOLDE_MOYEN_SUR_EPARGNE`). On les associe à la main.


In [ ]:
from utils.exploration import unifier_joints

TRONQUES = {
    "RATIO_SOLDE_MOYEN_SUR_EPAR": "RATIO_SOLDE_MOYEN_SUR_EPARGNE",
    "RATIO_NBJDEB_NONAUTO_SUR_NBJ": "RATIO_NBJDEB_NONAUTO_SUR_NBJDEB",
}
base_perim, rapport_joints = unifier_joints(
    base_perim, TRONQUES, max_seul=("CODITDBDF_PAR", "CODITDBDF_PRO", "cod_axe_unite"))
print(rapport_joints)
print(base_perim.select(pl.col(VARS_TEST).null_count()))

**Limites** :
- Pour les montants (encours, épargne…), la médiane d'un couple correspond à la moyenne des deux membres, donc elle sous-estime ce que possède le foyer : un total serait plus parlant.
- La CSP (`CODACVPRO_modif`) n'a pas de version agrégée : elle reste manquante pour les dossiers joints, qui formeront une modalité à part.
- L'impact reste limité, puisque les dossiers joints représentent environ 12 % des défauts.

**Piste pour la suite (feature engineering)** : construire des variables propres aux dossiers joints à partir des `_min`, `_max` et `_med`, par exemple le total du foyer (médiane × nombre de membres, exact pour un couple) ou l'écart entre les membres (max − min), qui peut révéler un membre en difficulté.


# Partie 2 — Segmentation

## 2.1 Critères de choix

On cherche à découper la population en quelques groupes qui auront chacun leur propre modèle. Une variable très liée au défaut n'est pas forcément une bonne variable de segmentation, donc on a retenu quatre critères pour juger les candidats.

1. La variable doit avoir un sens métier et être stable dans le temps. 
2. Les segments doivent avoir des moteurs de risque différents. Si les mêmes variables expliquent le défaut dans chaque groupe, un seul modèle qui contient la variable fera aussi bien que deux modèles séparés. Un simple écart de taux de défaut entre les groupes ne suffit donc pas.
3. Chaque segment doit contenir assez de défauts, au moins quelques centaines, pour qu'on puisse estimer un modèle dessus. 
4. La variable ne doit pas être le meilleur risk driver. Dans ce cas, il vaut mieux la garder comme variable explicative du modèle, sinon on perd son pouvoir prédictif à l'intérieur de chaque segment.


In [ ]:
from utils.segmentation import construire_candidats

cand = construire_candidats(base_perim)
print(cand.group_by("segment").agg(n=pl.len(), n_defaut=pl.col(TARGET).sum(), taux=pl.col(TARGET).mean()))

## 2.2 Candidats testés un par un

In [ ]:
from utils.segmentation import rank_segmentation_candidates

CANDIDATS = ["has_engagement", "type_credit", "has_revolving", "has_decouvert", "joint", "anciennete",
             "compte_inactif", "historique_defaut", "flux_pro", "patrimonial", "age", "CODACVPRO_modif"]
print(rank_segmentation_candidates(cand, CANDIDATS, TARGET))

In [ ]:
from utils.exploration import risk_rate_by_modality

for v in CANDIDATS:
    print(risk_rate_by_modality(cand, v, TARGET).sort(v))

Le tableau classe les candidats selon leur lien avec le défaut (V de Cramer). Ce classement ne suffit pas pour choisir : une variable très liée au défaut est souvent un bon risk driver plutôt qu'une bonne variable de segmentation (critère 4). On regarde donc aussi le nombre de défauts par modalité et le sens métier.

Candidats écartés :

| Candidat | Pourquoi on l'écarte |
|---|---|
| `historique_defaut` | c'est la variable la plus liée au défaut (6 % de défaut contre 0,75 % pour les autres), mais elle ne concerne que 2,4 % des dossiers et 374 défauts. C'est un risk driver, qu'on gardera dans le modèle (critère 4) |
| `patrimonial` | seulement 31 défauts (critère 3) |
| `flux_pro` | manquant pour presque 99 % des dossiers |
| `joint` | 284 défauts, et les dossiers joints sont déjà traités en 1.5 |
| `CODACVPRO_modif` (CSP) | 59 modalités, dont plusieurs sans aucun défaut : il faudra les regrouper au binning, ça ne peut pas servir de coupure |
| `has_revolving` | 214 défauts et 6 % des dossiers seulement (critère 3) |
| `has_decouvert` | même lien avec le défaut que `has_engagement`, et les deux se recoupent beaucoup (le découvert est un type d'engagement). On préfère `has_engagement`, qui couvre toutes les formes de crédit et se comprend mieux côté métier |
| `anciennete`, `age` | le risque baisse nettement avec l'ancienneté (de 3 % pour moins d'un an à 0,7 % au-delà de 5 ans) et avec l'âge, mais on verra en 2.4 que ça joue sur le niveau de risque et pas sur les moteurs : ce seront des variables du modèle |
| `compte_inactif` | aucun lien avec le défaut (0,88 % contre 0,87 %, p = 0,95) |
| crédit immobilier seul | 121 défauts (critère 3) |

On garde pour la suite `has_engagement` et le type de crédit : même s'ils sont moins liés au défaut que certaines variables, ils séparent des clients qui n'ont pas du tout la même relation avec la banque (avec ou sans crédit, et quel type de crédit).


## 2.3 Croisements de variables par arbre CART
Les tests univariés ne voient pas les croisements. Un arbre peu profond (profondeur 2 à 3, feuilles ≥ 5 % de la population) propose des découpages multivariés ; chaque feuille est un segment candidat. On le lance sur les variables structurelles seules, puis en ajoutant des risk drivers forts.

In [ ]:
from utils.segmentation import tree_segmentation

STRUCTURELLES = ["has_engagement", "type_credit", "has_decouvert", "has_revolving", "joint",
                 "compte_inactif", "patrimonial", "anciennete_mois"]
AVEC_DRIVERS = STRUCTURELLES + ["historique_defaut", "age_ans"]

for nom, feats in [("structurelles", STRUCTURELLES), ("avec risk drivers", AVEC_DRIVERS)]:
    for prof in (2, 3):
        feuilles, _ = tree_segmentation(cand, feats, TARGET, max_depth=prof, min_leaf_share=0.05)
        print(f"=== {nom} | profondeur {prof}")
        print(feuilles)

Sur le périmètre final, la première coupure de l'arbre n'est pas `has_engagement` mais l'ancienneté, autour de 13,5 ans (162 mois). L'arbre sépare ensuite selon le type de crédit : parmi les clients récents, ceux qui n'ont que des facilités (découvert, carte à débit différé) sont les plus risqués ; parmi les anciens, ce sont ceux qui ont un crédit conso. Les écarts sont forts : 3,7 % de défaut pour les clients de moins de 5 ans qui n'ont que des facilités, contre 0,2 % pour les clients de plus de 32 ans sans crédit conso.

Ajouter l'âge et l'historique de défaut ne change rien aux arbres. L'historique de défaut ne peut de toute façon pas être utilisé ici, car il concerne moins de 5 % des dossiers, la taille minimale imposée aux feuilles.

Lors d'un premier passage, avant d'exclure les mineurs, la première coupure était `has_engagement` : elle isolait surtout les mineurs, presque tous sans crédit et quasiment sans défaut. Une fois ceux-ci retirés, c'est l'ancienneté qui explique le mieux les écarts de risque.

L'arbre ne regarde que le niveau de risque dans chaque feuille. Il nous dit que l'ancienneté et le type de crédit sont les variables qui font le plus varier le taux de défaut, mais pas si les moteurs du risque changent d'un groupe à l'autre. C'est ce qu'on teste en 2.4.


## 2.4 Test des moteurs de risque : IV par segment
Deux segments justifient deux modèles seulement si **les variables qui prédisent le défaut diffèrent**. Pour chaque découpage candidat, on calcule l'IV de toutes les variables dans chaque segment (découpage fin en 10 quantiles, missing = classe à part, seuils de Siddiqi), puis on compare les classements :
- corrélation de Spearman des IV entre segments ;
- nombre de variables communes aux deux top 15.

**Règle de décision** : corrélation > 0,8 et plus de 10 variables communes → mêmes moteurs → pas de séparation.

Variables exclues de l'analyse : identifiant, cible, dates brutes (à transformer en durées au ch. 2.4 du cours), variables de fuite (1.2), variables constantes (1.4), et provisoirement les qualitatives à plus de 15 modalités, dont l'IV est gonflée tant qu'elles ne sont pas regroupées.

In [ ]:
from utils.binning import iv_ranking

EXCLURE = {ID, TARGET, DATE, "SelectionProb", "SamplingWeight", "type_PM_PP", "code_marche_v2"}
VARS = [c for c, t in base_perim.schema.items() if c not in EXCLURE and not t.is_temporal()]

iv_glob = iv_ranking(cand, VARS, TARGET)
VARS_SEG = iv_glob.filter(pl.col("n_classes") <= 15)["variable"].to_list()
print("écartées provisoirement (> 15 classes) :", iv_glob.filter(pl.col("n_classes") > 15)["variable"].to_list())

### Découpage par type de crédit : sans engagement / facilités seules / crédit amortissable

In [ ]:
from utils.segmentation import risk_rate_over_time, comparer_classements
from utils.binning import iv_by_segment

print(risk_rate_over_time(cand, "seg_type_credit", TARGET, DATE))
iv_type_credit = iv_by_segment(cand, VARS_SEG, TARGET, "seg_type_credit")
print(comparer_classements(iv_type_credit))

**Constat (type de crédit)** : les dossiers qui n'ont que des facilités (S2) et ceux qui ont un crédit amortissable (S3) ont des taux de défaut proches, qui se croisent en février, et presque les mêmes variables en tête (corrélation 0,81, 13 variables communes sur 15). Avoir un crédit amortissable ne change pas vraiment ce qui prédit le défaut, donc on regroupe S2 et S3.


### Découpage par ancienneté dans le segment avec engagement

In [ ]:
from utils.segmentation import risk_rate_over_time, comparer_classements
from utils.binning import iv_by_segment

print(risk_rate_over_time(cand, "seg_anciennete", TARGET, DATE))
iv_anciennete = iv_by_segment(cand, VARS_SEG, TARGET, "seg_anciennete")
print(comparer_classements(iv_anciennete))

**Constat (ancienneté)** : les clients avec engagement depuis moins de 5 ans font environ trois fois plus défaut que les plus anciens (2,4 % contre 0,8 %), et cet écart est stable tous les mois. Par contre, ce sont les mêmes variables qui prédisent le défaut dans les deux groupes (corrélation 0,91, 12 variables communes sur 15). L'ancienneté joue donc sur le niveau de risque mais pas sur les moteurs : on la garde comme variable du modèle plutôt que d'en faire un segment.


### Découpage retenu : sans engagement / avec engagement

In [ ]:
from utils.segmentation import risk_rate_over_time, comparer_classements
from utils.binning import iv_by_segment

iv_segment = iv_by_segment(cand, VARS_SEG, TARGET, "segment")
print(comparer_classements(iv_segment))
print(iv_segment.head(20))

**Constat (sans / avec engagement)** : la corrélation entre les deux classements est de 0,74, avec 10 variables communes sur 15. On est en dessous de notre seuil de fusion, mais pas de beaucoup. Les variables de solde et de jours débiteurs sont importantes dans les deux groupes. Les différences viennent surtout de deux familles de variables :
- l'historique de défaut (`Dfo_max_*`, ancienneté dans le sain) est beaucoup plus discriminant sans engagement (IV autour de 0,5-0,6) qu'avec engagement (autour de 0,1). Sans crédit en cours, le passé du client est une des seules informations sur son risque ;
- les variables liées au crédit (arriérés, encours bilan et hors bilan) n'existent que pour les dossiers avec engagement.


## 2.5 Décision et stabilité

In [ ]:
from utils.segmentation import risk_rate_over_time

print(risk_rate_over_time(cand, "segment", TARGET, DATE))
print(cand.group_by(DATE, "segment").agg(n=pl.len(), n_defaut=pl.col(TARGET).sum()).sort(DATE, "segment"))

On retient **deux segments** : SA, les dossiers sans engagement (`has_engagement = 0`), et SB, les dossiers avec engagement (`has_engagement = 1`). SA regroupe environ 23 500 dossiers par mois (36 %) et 515 défauts au total, SB environ 42 400 dossiers par mois (64 %) et 1 779 défauts.

Ce qui justifie ce choix :
- Côté métier, la séparation est simple : le client a une relation de crédit avec la banque ou non.
- Les moteurs de risque ne sont pas tout à fait les mêmes. L'historique de défaut compte beaucoup plus dans SA, et les variables de crédit (arriérés, encours) n'existent que dans SB. La différence reste modérée (corrélation de 0,74 entre les classements de variables).
- SA est moins risqué que SB tous les mois : SB fait entre 1,4 et 2,5 fois plus défaut selon le mois, donc toujours plus de 30 % d'écart.

Les autres découpages testés ont été écartés : le type de crédit et l'ancienneté changent le niveau de risque mais pas les variables qui l'expliquent, et les autres candidats n'ont pas assez de défauts. Avec environ 2 300 défauts, on ne peut pas raisonnablement aller au-delà de deux segments.

Cependant attention :
- Le nombre de défauts de SA augmente fortement en avril (182 contre 100 à 122 les autres mois), alors que SB reste stable. Avril étant l'échantillon OOT, le modèle SA sera testé sur un mois un peu atypique, ce qu'il faudra prendre en compte au calibrage.
- SA est le plus petit segment : sans avril, il reste environ 330 défauts pour l'apprentissage. C'est suffisant pour une régression logistique, mais avec un nombre limité de variables.

Comme la différence entre SA et SB n'est pas énorme, ce choix reste à confirmer au moment de la modélisation. On comparera le Gini d'un modèle unique, avec `has_engagement` comme variable, à celui des deux modèles séparés. Si les deux modèles n'apportent presque rien, on reviendra à un modèle unique.


In [ ]:
from utils.exploration import controle_groupes, risk_rate_by_modality, exporter_excel
from utils.segmentation import rank_segmentation_candidates, tree_segmentation, comparer_classements, risk_rate_over_time

defaut_a_date, _ = controle_groupes(base, {
    "arr_90j": pl.col("NBJ_ARR_SS") >= 90,
    "dfo_non_resolu": pl.col("DATE_DERNIER_DFO") >= pl.col("DATE_SAIN"),
    "date_sain_null": pl.col("DATE_SAIN").is_null(),
    "dfo_M1_M3": pl.col("Dfo_max_Mm1_Mm3") == 1,
}, TARGET)

arbres = pl.concat([
    tree_segmentation(cand, STRUCTURELLES, TARGET, max_depth=prof, min_leaf_share=0.05)[0]
    .with_columns(profondeur=pl.lit(prof))
    for prof in (2, 3)
])

comparaisons = pl.concat([
    comparer_classements(iv_type_credit).with_columns(decoupage=pl.lit("type de crédit")),
    comparer_classements(iv_anciennete).with_columns(decoupage=pl.lit("ancienneté")),
    comparer_classements(iv_segment).with_columns(decoupage=pl.lit("retenu : sans / avec engagement")),
])

volumes_segments = (
    cand.group_by(DATE, "segment")
    .agg(n=pl.len(), n_defaut=pl.col(TARGET).sum(), taux_defaut=pl.col(TARGET).mean())
    .sort(DATE, "segment")
)

exporter_excel({
    "1_Perimetre": suivi_perimetre,
    "1_Defaut_a_date": defaut_a_date,
    "1_Cible_par_mois": risk_rate_by_modality(base, DATE, TARGET).sort(DATE),
    "2_Candidats": rank_segmentation_candidates(cand, CANDIDATS, TARGET),
    "2_Arbres_CART": arbres,
    "2_Comparaison_IV": comparaisons,
    "2_IV_SA_vs_SB": iv_segment,
    "2_Stabilite_taux": risk_rate_over_time(cand, "segment", TARGET, DATE),
    "2_Stabilite_volumes": volumes_segments,
}, "outputs/segmentation.xlsx")
print("export : outputs/segmentation.xlsx")


In [ ]:
SORTIE = Path("data/processed/base_segmentee.parquet")
cand.select(*base_perim.columns, "segment").write_parquet(SORTIE)
print(SORTIE, cand.shape[0], "lignes")
